<a href="https://colab.research.google.com/github/lynnfdsouza/CUAS21/blob/main/Drone_Detection_targeting_system_single_multi_swarm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install ultralytics pyserial

In [16]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
import serial
from google.colab.patches import cv2_imshow

# System parameters
MAX_RANGE = 100  # Detection range in meters
FOV_HORIZONTAL = 180  # Camera field of view in degrees
MIN_ANGLE = -90  # Leftmost pan angle
MAX_ANGLE = 90   # Rightmost pan angle
ANGLE_TOLERANCE = 5  # Degrees tolerance for targeting

# YOLOv8 parameters
CONFIDENCE_THRESHOLD = 0.5  # Minimum confidence for detection
KNOWN_DRONE_WIDTH = 0.5  # Known drone width in meters (for distance estimation)
FOCAL_LENGTH = 800  # Camera focal length in pixels (calibrate for your camera)

# Target classes (COCO dataset class IDs)
# You can train a custom model specifically for drones
TARGET_CLASSES = ['bird', 'airplane', 'kite']  # Proxy classes, or use custom drone model

class YOLODroneDetector:
    def __init__(self, model_path='yolov8n.pt', camera_index=0):
        """
        Initialize YOLOv8 drone detector.

        Args:
            model_path: Path to YOLOv8 model (yolov8n.pt, yolov8s.pt (small), yolov8m.pt (medium), etc.)
                       Or path to custom trained drone detection model
            camera_index: Camera device index
        """
        print("Loading YOLOv8 model...")
        self.model = YOLO(model_path)
        self.cap = cv2.VideoCapture(camera_index)
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_center_x = self.frame_width // 2
        print(f"Model loaded. Camera resolution: {self.frame_width}x{self.frame_height}")

    def estimate_distance(self, bbox_width):
        """Estimate distance to drone using similar triangles."""
        if bbox_width == 0:
            return MAX_RANGE
        distance = (KNOWN_DRONE_WIDTH * FOCAL_LENGTH) / bbox_width
        return min(distance, MAX_RANGE)

    def pixel_to_angle(self, pixel_x):
        """Convert pixel x-coordinate to angle relative to center."""
        offset = pixel_x - self.frame_center_x
        angle = (offset / self.frame_center_x) * (FOV_HORIZONTAL / 2)
        return angle

    def detect_drones(self):
        """Detect drones using YOLOv8."""
        ret, frame = self.cap.read()
        if not ret:
            return [], None  # Return None for display_frame if frame read fails

        # Run YOLOv8 inference
        results = self.model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)

        # Create annotated frame
        display_frame = results[0].plot()

        drones = []

        # Process detections
        for result in results:
            boxes = result.boxes
            for box in boxes:
                # Get class name
                cls_id = int(box.cls[0])
                class_name = self.model.names[cls_id]
                confidence = float(box.conf[0])

                # Check if detected object is a target class
                if class_name in TARGET_CLASSES:
                    # Get bounding box coordinates
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                    # Calculate center and width
                    center_x = int((x1 + x2) / 2)
                    center_y = int((y1 + y2) / 2)
                    bbox_width = x2 - x1

                    # Calculate angle and distance
                    angle = self.pixel_to_angle(center_x)
                    distance = self.estimate_distance(bbox_width)

                    # Only consider drones in valid angle range
                    if MIN_ANGLE <= angle <= MAX_ANGLE:
                        drones.append({
                            'distance': distance,
                            'angle': angle,
                            'confidence': confidence,
                            'class': class_name,
                            'bbox': (int(x1), int(y1), int(x2), int(y2)),
                            'center': (center_x, center_y)
                        })

                        # Add targeting info to display
                        cv2.circle(display_frame, (center_x, center_y), 5, (0, 0, 255), -1)
                        cv2.line(display_frame, (self.frame_center_x, center_y),
                                (center_x, center_y), (255, 0, 0), 2)

                        info_text = f"{distance:.1f}m | {angle:.1f}deg | {confidence:.2f}"
                        cv2.putText(display_frame, info_text,
                                   (int(x1), int(y1) - 25),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        return drones, display_frame

    def release(self):
        """Release resources."""
        self.cap.release()
        cv2.destroyAllWindows()

def calculate_target_angle(drone_angle):
    """Ensure the bazooka pans within safe limits."""
    if MIN_ANGLE <= drone_angle <= MAX_ANGLE:
        return drone_angle
    else:
        return max(MIN_ANGLE, min(MAX_ANGLE, drone_angle))

def aim_bazooka(drone_distance, drone_angle):
    """Aim the bazooka at the target."""
    target_angle = calculate_target_angle(drone_angle)
    print(f"  → Aiming bazooka: {drone_distance:.2f}m, {target_angle:.2f}°")

    # In real system: send commands to servo controller
    # Example using serial communication:
    try:
        servo = serial.Serial('/dev/ttyUSB0', 9600)
        servo.write(f"ANGLE:{target_angle}\n".encode())
        servo.close()
    except serial.SerialException as e:
        print(f"  ⚠ Serial communication error: {e}")


    time.sleep(0.1)
    return abs(target_angle - drone_angle) < ANGLE_TOLERANCE

def fire_bazooka():
    """Fire the bazooka."""
    print("  🚀 FIRING BAZOOKA!")

    # In real system: trigger firing mechanism
    # Example using GPIO (Raspberry Pi):
    # GPIO.output(FIRE_PIN, GPIO.HIGH)
    # time.sleep(0.1)
    # GPIO.output(FIRE_PIN, GPIO.LOW)

    time.sleep(0.2)
    return True

def prioritize_targets(drones):
    """
    Prioritize targets based on threat level.
    Closer drones are higher priority.
    """
    return sorted(drones, key=lambda d: d['distance'])

def main():
    print("=" * 60)
    print("YOLOv8 DRONE DETECTION AND TARGETING SYSTEM")
    print("=" * 60)
    print(f"Pan range: {MIN_ANGLE}° to {MAX_ANGLE}°")
    print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
    print(f"Target classes: {', '.join(TARGET_CLASSES)}")
    print("\nPress 'q' to quit")
    print("=" * 60)

    # Initialize detector with YOLOv8
    # Options: yolov8n.pt (nano), yolov8s.pt (small), yolov8m.pt (medium)
    # Or use custom trained model: 'path/to/your/drone_model.pt'
    detector = YOLODroneDetector(model_path='yolov8n.pt', camera_index=0)

    frame_count = 0
    fps_start_time = time.time()

    try:
        while True:
            # Detect drones using YOLOv8
            drones, display_frame = detector.detect_drones()

            # Calculate FPS
            frame_count += 1
            if frame_count % 30 == 0:
                fps = 30 / (time.time() - fps_start_time)
                fps_start_time = time.time()
            else:
                fps = 0

            # Draw UI elements
            if display_frame is not None: # Only draw if frame is valid
                cv2.line(display_frame, (detector.frame_center_x, 0),
                        (detector.frame_center_x, detector.frame_height), (255, 0, 0), 2)

                # Status overlay
                status_text = f"Targets: {len(drones)}"
                if fps > 0:
                    status_text += f" | FPS: {fps:.1f}"

                cv2.rectangle(display_frame, (0, 0), (400, 60), (0, 0, 0), -1)
                cv2.putText(display_frame, status_text, (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(display_frame, "YOLOv8 Drone Detector", (10, 55),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

                # Display frame
                cv2_imshow(display_frame)

            if not drones:
                if frame_count % 30 == 0:  # Print every 30 frames
                    print("Scanning... No targets detected")
            else:
                # Prioritize closest targets
                prioritized = prioritize_targets(drones)

                print(f"\n[DETECTION] {len(drones)} target(s) detected:")
                for i, drone in enumerate(prioritized, 1):
                    print(f"  Target {i}: {drone['class']} - "
                          f"{drone['distance']:.2f}m, {drone['angle']:.2f}°, "
                          f"conf: {drone['confidence']:.2f}")

                    # Engage highest priority target
                    if i == 1:
                        if aim_bazooka(drone['distance'], drone['angle']):
                            fire_bazooka()
                        else:
                            print("  ⚠ Target out of safe range")

            # Check for quit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except KeyboardInterrupt:
        print("\n\nShutting down targeting system...")
    finally:
        detector.release()
        print("System offline.")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No targets detected
Scanning... No target

In [18]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
from collections import deque, defaultdict
import serial
from google.colab.patches import cv2_imshow

# System parameters
MAX_RANGE = 100  # Detection range in meters
FOV_HORIZONTAL = 180  # Camera field of view in degrees
MIN_ANGLE = -90  # Leftmost pan angle
MAX_ANGLE = 90   # Rightmost pan angle
ANGLE_TOLERANCE = 5  # Degrees tolerance for targeting

# YOLOv8 parameters
CONFIDENCE_THRESHOLD = 0.5  # Minimum confidence for detection
KNOWN_DRONE_WIDTH = 0.5  # Known drone width in meters
FOCAL_LENGTH = 800  # Camera focal length in pixels

# Tracking parameters
MAX_TRACKING_DISTANCE = 100  # Max pixels to associate detections
TRACK_HISTORY_LENGTH = 30  # Number of frames to keep in history
MIN_TRACK_FRAMES = 5  # Minimum frames before engaging target
COOLDOWN_TIME = 3.0  # Seconds before re-engaging same target

# Multi-target parameters
MAX_SIMULTANEOUS_TARGETS = 3  # Maximum targets to track simultaneously
ENGAGEMENT_PRIORITY = 'distance'  # 'distance', 'confidence', or 'velocity'

# Target classes
TARGET_CLASSES = ['bird', 'airplane', 'kite']

class DroneTracker:
    """Track multiple drones across frames."""

    def __init__(self):
        self.tracks = {}  # track_id -> track data
        self.next_track_id = 0
        self.engaged_targets = {}  # track_id -> last engagement time

    def update(self, detections):
        """Update tracks with new detections."""
        if not detections:
            # Age out old tracks
            self._age_tracks()
            return self.tracks

        # Get current track positions
        current_positions = {tid: track['positions'][-1] for tid, track in self.tracks.items()
                           if track['active']}

        # Associate detections with existing tracks
        matched_tracks = set()
        matched_detections = set()

        for det_idx, detection in enumerate(detections):
            det_center = detection['center']
            best_match = None
            min_distance = MAX_TRACKING_DISTANCE

            for track_id, track_pos in current_positions.items():
                if track_id in matched_tracks:
                    continue

                distance = np.linalg.norm(np.array(det_center) - np.array(track_pos))
                if distance < min_distance:
                    min_distance = distance
                    best_match = track_id

            if best_match is not None:
                # Update existing track
                self._update_track(best_match, detection)
                matched_tracks.add(best_match)
                matched_detections.add(det_idx)
            else:
                # Create new track
                self._create_track(detection)
                matched_detections.add(det_idx)

        # Mark unmatched tracks as inactive
        for track_id in list(self.tracks.keys()):
            if self.tracks[track_id]['active']:
                self.tracks[track_id]['frames_since_update'] += 1
                if self.tracks[track_id]['frames_since_update'] > 10:
                    self.tracks[track_id]['active'] = False

        return self.tracks

    def _create_track(self, detection):
        """Create a new track."""
        track_id = self.next_track_id
        self.next_track_id += 1

        self.tracks[track_id] = {
            'positions': deque([detection['center']], maxlen=TRACK_HISTORY_LENGTH),
            'distances': deque([detection['distance']], maxlen=TRACK_HISTORY_LENGTH),
            'angles': deque([detection['angle']], maxlen=TRACK_HISTORY_LENGTH),
            'confidences': deque([detection['confidence']], maxlen=TRACK_HISTORY_LENGTH),
            'class': detection['class'],
            'bbox': detection['bbox'],
            'frames_tracked': 1,
            'frames_since_update': 0,
            'active': True,
            'velocity': 0.0,
            'last_detection': detection
        }

    def _update_track(self, track_id, detection):
        """Update existing track."""
        track = self.tracks[track_id]

        # Calculate velocity (pixels per frame)
        if len(track['positions']) > 0:
            old_pos = np.array(track['positions'][-1])
            new_pos = np.array(detection['center'])
            track['velocity'] = np.linalg.norm(new_pos - old_pos)

        track['positions'].append(detection['center'])
        track['distances'].append(detection['distance'])
        track['angles'].append(detection['angle'])
        track['confidences'].append(detection['confidence'])
        track['bbox'] = detection['bbox']
        track['frames_tracked'] += 1
        track['frames_since_update'] = 0
        track['active'] = True
        track['last_detection'] = detection

    def _age_tracks(self):
        """Age out tracks that haven't been updated."""
        for track_id in list(self.tracks.keys()):
            if self.tracks[track_id]['active']:
                self.tracks[track_id]['frames_since_update'] += 1
                if self.tracks[track_id]['frames_since_update'] > 10:
                    self.tracks[track_id]['active'] = False

    def can_engage(self, track_id):
        """Check if target can be engaged."""
        track = self.tracks[track_id]

        # Need minimum frames for stable tracking
        if track['frames_tracked'] < MIN_TRACK_FRAMES:
            return False

        # Check cooldown period
        if track_id in self.engaged_targets:
            time_since_engagement = time.time() - self.engaged_targets[track_id]
            if time_since_engagement < COOLDOWN_TIME:
                return False

        return True

    def mark_engaged(self, track_id):
        """Mark target as engaged."""
        self.engaged_targets[track_id] = time.time()

    def get_priority_targets(self, max_targets=MAX_SIMULTANEOUS_TARGETS):
        """Get priority targets for engagement."""
        engageable = []

        for track_id, track in self.tracks.items():
            if not track['active']:
                continue
            if not self.can_engage(track_id):
                continue

            # Get latest values
            detection = track['last_detection']

            # Calculate priority score
            if ENGAGEMENT_PRIORITY == 'distance':
                priority = -detection['distance']  # Closer is higher priority
            elif ENGAGEMENT_PRIORITY == 'confidence':
                priority = detection['confidence']
            elif ENGAGEMENT_PRIORITY == 'velocity':
                priority = track['velocity']
            else:
                priority = -detection['distance']

            engageable.append({
                'track_id': track_id,
                'priority': priority,
                'detection': detection,
                'track': track
            })

        # Sort by priority and return top N
        engageable.sort(key=lambda x: x['priority'], reverse=True)
        return engageable[:max_targets]

class YOLODroneDetector:
    def __init__(self, model_path='yolov8n.pt', camera_index=0):
        """Initialize YOLOv8 drone detector with tracking."""
        print("Loading YOLOv8 model...")
        self.model = YOLO(model_path)
        self.cap = cv2.VideoCapture(camera_index)
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_center_x = self.frame_width // 2
        self.tracker = DroneTracker()
        print(f"Model loaded. Camera: {self.frame_width}x{self.frame_height}")

    def estimate_distance(self, bbox_width):
        """Estimate distance using similar triangles."""
        if bbox_width == 0:
            return MAX_RANGE
        distance = (KNOWN_DRONE_WIDTH * FOCAL_LENGTH) / bbox_width
        return min(distance, MAX_RANGE)

    def pixel_to_angle(self, pixel_x):
        """Convert pixel x-coordinate to angle."""
        offset = pixel_x - self.frame_center_x
        angle = (offset / self.frame_center_x) * (FOV_HORIZONTAL / 2)
        return angle

    def detect_and_track(self):
        """Detect and track drones."""
        ret, frame = self.cap.read()
        if not ret:
            return [], None # Return None for display_frame if frame read fails

        # Run YOLOv8 inference
        results = self.model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        display_frame = frame.copy()

        detections = []

        # Process YOLO detections
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls[0])
                class_name = self.model.names[cls_id]
                confidence = float(box.conf[0])

                if class_name in TARGET_CLASSES:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    center_x = int((x1 + x2) / 2)
                    center_y = int((y1 + y2) / 2)
                    bbox_width = x2 - x1

                    angle = self.pixel_to_angle(center_x)
                    distance = self.estimate_distance(bbox_width)

                    if MIN_ANGLE <= angle <= MAX_ANGLE:
                        detections.append({
                            'distance': distance,
                            'angle': angle,
                            'confidence': confidence,
                            'class': class_name,
                            'bbox': (int(x1), int(y1), int(x2), int(y2)),
                            'center': (center_x, center_y)
                        })

        # Update tracker
        tracks = self.tracker.update(detections)

        # Draw tracks
        for track_id, track in tracks.items():
            if not track['active']:
                continue

            # Draw track history
            points = list(track['positions'])
            for i in range(1, len(points)):
                thickness = int(np.sqrt(TRACK_HISTORY_LENGTH / float(i + 1)) * 2)
                cv2.line(display_frame, points[i-1], points[i], (0, 255, 255), thickness)

            # Draw current bounding box
            x1, y1, x2, y2 = track['bbox']
            color = (0, 255, 0) if self.tracker.can_engage(track_id) else (0, 165, 255)
            cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)

            # Draw track info
            center = track['positions'][-1]
            detection = track['last_detection']

            info_text = f"ID:{track_id} | {detection['distance']:.1f}m | {detection['angle']:.1f}°"
            cv2.putText(display_frame, info_text, (x1, y1 - 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            status_text = f"Frames:{track['frames_tracked']} | V:{track['velocity']:.1f}"
            cv2.putText(display_frame, status_text, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

            # Draw crosshair on target
            cv2.drawMarker(display_frame, center, color, cv2.MARKER_CROSS, 20, 2)

        return tracks, display_frame

    def release(self):
        """Release resources."""
        self.cap.release()
        cv2.destroyAllWindows()

def aim_and_track(target_info):
    """Aim at moving target with predictive tracking."""
    detection = target_info['detection']
    track = target_info['track']

    # Predict future position based on velocity
    if len(track['positions']) >= 2:
        positions = list(track['positions'])
        recent_velocity = np.array(positions[-1]) - np.array(positions[-2])
        # Predict 5 frames ahead
        predicted_pos = np.array(positions[-1]) + (recent_velocity * 5)

        print(f"  → Tracking ID:{target_info['track_id']} with predictive aim")
        print(f"    Position: {detection['distance']:.2f}m, {detection['angle']:.2f}°")
        print(f"    Velocity: {track['velocity']:.1f} px/frame")

    time.sleep(0.1)
    return True

def fire_bazooka(track_id):
    """Fire at tracked target."""
    print(f"  🚀 ENGAGING TARGET ID:{track_id}!")
    time.sleep(0.2)
    return True

def main():
    print("=" * 70)
    print("YOLOv8 MULTI-TARGET TRACKING & ENGAGEMENT SYSTEM")
    print("=" * 70)
    print(f"Tracking: {TRACK_HISTORY_LENGTH} frames | Min track: {MIN_TRACK_FRAMES}")
    print(f"Max targets: {MAX_SIMULTANEOUS_TARGETS} | Priority: {ENGAGEMENT_PRIORITY}")
    print(f"Cooldown: {COOLDOWN_TIME}s | Angle tolerance: {ANGLE_TOLERANCE}°")
    print("\nPress 'q' to quit")
    print("=" * 70)

    detector = YOLODroneDetector(model_path='yolov8n.pt', camera_index=0)

    frame_count = 0
    fps_start = time.time()
    fps = 0

    try:
        while True:
            # Detect and track
            tracks, display_frame = detector.detect_and_track()

            # Calculate FPS
            frame_count += 1
            if frame_count % 30 == 0:
                fps = 30 / (time.time() - fps_start)
                fps_start = time.time()

            # Draw UI
            if display_frame is not None: # Only draw if frame is valid
              cv2.line(display_frame, (detector.frame_center_x, 0),
                      (detector.frame_center_x, detector.frame_height), (255, 0, 0), 2)

              # Check if tracks is a dictionary before accessing values()
              if isinstance(tracks, dict):
                  active_tracks = sum(1 for t in tracks.values() if t['active'])
              else:
                  active_tracks = 0 # Or handle appropriately if tracks is a list

              # Status overlay
              cv2.rectangle(display_frame, (0, 0), (500, 90), (0, 0, 0), -1)
              cv2.putText(display_frame, f"Active Tracks: {active_tracks} | FPS: {fps:.1f}",
                         (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
              cv2.putText(display_frame, f"Priority: {ENGAGEMENT_PRIORITY.upper()}",
                         (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
              cv2.putText(display_frame, "YOLOv8 Multi-Target System",
                         (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

              cv2_imshow(display_frame)

            # Get priority targets
            if isinstance(tracks, dict):
                priority_targets = detector.tracker.get_priority_targets()
            else:
                priority_targets = []


            if priority_targets:
                print(f"\n[ENGAGEMENT QUEUE] {len(priority_targets)} target(s):")
                for i, target in enumerate(priority_targets, 1):
                    det = target['detection']
                    print(f"  {i}. ID:{target['track_id']} - {det['class']} | "
                          f"{det['distance']:.2f}m, {det['angle']:.2f}° | "
                          f"Priority: {target['priority']:.2f}")

                    # Engage target
                    if aim_and_track(target):
                        fire_bazooka(target['track_id'])
                        detector.tracker.mark_engaged(target['track_id'])
            else:
                if frame_count % 30 == 0:
                    print("Scanning... No engageable targets")

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except KeyboardInterrupt:
        print("\n\nShutting down...")
    finally:
        detector.release()
        print("System offline.")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targe

# Task
Add swarm detection and engagement capabilities to the drone tracking system.

## Define swarm criteria

### Subtask:
Determine what constitutes a "swarm" based on factors like proximity, number of objects, and movement coherence.


**Reasoning**:
Define the constants for swarm detection criteria.



In [19]:
# Swarm detection parameters
SWARM_MIN_MEMBERS = 3  # Minimum number of tracked objects to be considered a swarm
SWARM_MAX_DISTANCE = 10  # Maximum distance between any two members for swarm inclusion
# Future enhancement: Add criteria for movement coherence (similar velocity vectors)

## Implement swarm detection logic

### Subtask:
Add functions or methods to the `DroneTracker` class or a new class to identify potential swarms based on the defined criteria. This might involve spatial clustering or analyzing movement patterns.


**Reasoning**:
Add the `find_swarms` method to the `DroneTracker` class to identify potential swarms based on proximity and minimum members.



In [22]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
from collections import deque, defaultdict
import serial
from google.colab.patches import cv2_imshow

# System parameters
MAX_RANGE = 100  # Detection range in meters
FOV_HORIZONTAL = 180  # Camera field of view in degrees
MIN_ANGLE = -90  # Leftmost pan angle
MAX_ANGLE = 90   # Rightmost pan angle
ANGLE_TOLERANCE = 5  # Degrees tolerance for targeting

# YOLOv8 parameters
CONFIDENCE_THRESHOLD = 0.5  # Minimum confidence for detection
KNOWN_DRONE_WIDTH = 0.5  # Known drone width in meters
FOCAL_LENGTH = 800  # Camera focal length in pixels

# Tracking parameters
MAX_TRACKING_DISTANCE = 100  # Max pixels to associate detections
TRACK_HISTORY_LENGTH = 30  # Number of frames to keep in history
MIN_TRACK_FRAMES = 5  # Minimum frames before engaging target
COOLDOWN_TIME = 3.0  # Seconds before re-engaging same target

# Multi-target parameters
MAX_SIMULTANEOUS_TARGETS = 3  # Maximum targets to track simultaneously
ENGAGEMENT_PRIORITY = 'distance'  # 'distance', 'confidence', or 'velocity'

# Target classes
TARGET_CLASSES = ['bird', 'airplane', 'kite']

# Swarm detection parameters
SWARM_MIN_MEMBERS = 3  # Minimum number of tracked objects to be considered a swarm
SWARM_MAX_DISTANCE = 10  # Maximum distance between any two members for swarm inclusion
# Future enhancement: Add criteria for movement coherence (similar velocity vectors)

class DroneTracker:
    """Track multiple drones across frames."""

    def __init__(self):
        self.tracks = {}  # track_id -> track data
        self.next_track_id = 0
        self.engaged_targets = {}  # track_id -> last engagement time

    def update(self, detections):
        """Update tracks with new detections."""
        if not detections:
            # Age out old tracks
            self._age_tracks()
            return self.tracks

        # Get current track positions
        current_positions = {tid: track['positions'][-1] for tid, track in self.tracks.items()
                           if track['active']}

        # Associate detections with existing tracks
        matched_tracks = set()
        matched_detections = set()

        for det_idx, detection in enumerate(detections):
            det_center = detection['center']
            best_match = None
            min_distance = MAX_TRACKING_DISTANCE

            for track_id, track_pos in current_positions.items():
                if track_id in matched_tracks:
                    continue

                distance = np.linalg.norm(np.array(det_center) - np.array(track_pos))
                if distance < min_distance:
                    min_distance = distance
                    best_match = track_id

            if best_match is not None:
                # Update existing track
                self._update_track(best_match, detection)
                matched_tracks.add(best_match)
                matched_detections.add(det_idx)
            else:
                # Create new track
                self._create_track(detection)
                matched_detections.add(det_idx)

        # Mark unmatched tracks as inactive
        for track_id in list(self.tracks.keys()):
            if self.tracks[track_id]['active']:
                self.tracks[track_id]['frames_since_update'] += 1
                if self.tracks[track_id]['frames_since_update'] > 10:
                    self.tracks[track_id]['active'] = False

        return self.tracks

    def _create_track(self, detection):
        """Create a new track."""
        track_id = self.next_track_id
        self.next_track_id += 1

        self.tracks[track_id] = {
            'positions': deque([detection['center']], maxlen=TRACK_HISTORY_LENGTH),
            'distances': deque([detection['distance']], maxlen=TRACK_HISTORY_LENGTH),
            'angles': deque([detection['angle']], maxlen=TRACK_HISTORY_LENGTH),
            'confidences': deque([detection['confidence']], maxlen=TRACK_HISTORY_LENGTH),
            'class': detection['class'],
            'bbox': detection['bbox'],
            'frames_tracked': 1,
            'frames_since_update': 0,
            'active': True,
            'velocity': 0.0,
            'last_detection': detection
        }

    def _update_track(self, track_id, detection):
        """Update existing track."""
        track = self.tracks[track_id]

        # Calculate velocity (pixels per frame)
        if len(track['positions']) > 0:
            old_pos = np.array(track['positions'][-1])
            new_pos = np.array(detection['center'])
            track['velocity'] = np.linalg.norm(new_pos - old_pos)

        track['positions'].append(detection['center'])
        track['distances'].append(detection['distance'])
        track['angles'].append(detection['angle'])
        track['confidences'].append(detection['confidence'])
        track['bbox'] = detection['bbox']
        track['frames_tracked'] += 1
        track['frames_since_update'] = 0
        track['active'] = True
        track['last_detection'] = detection


    def _age_tracks(self):
        """Age out tracks that haven't been updated."""
        for track_id in list(self.tracks.keys()):
            if self.tracks[track_id]['active']:
                self.tracks[track_id]['frames_since_update'] += 1
                if self.tracks[track_id]['frames_since_update'] > 10:
                    self.tracks[track_id]['active'] = False

    def can_engage(self, track_id):
        """Check if target can be engaged."""
        track = self.tracks[track_id]

        # Need minimum frames for stable tracking
        if track['frames_tracked'] < MIN_TRACK_FRAMES:
            return False

        # Check cooldown period
        if track_id in self.engaged_targets:
            time_since_engagement = time.time() - self.engaged_targets[track_id]
            if time_since_engagement < COOLDOWN_TIME:
                return False

        return True

    def mark_engaged(self, track_id):
        """Mark target as engaged."""
        self.engaged_targets[track_id] = time.time()


    def get_priority_targets(self, swarms, max_targets=MAX_SIMULTANEOUS_TARGETS):
        """Get priority targets for engagement, prioritizing swarms."""
        engageable = []
        swarm_targets = []

        # Add swarm members to engageable list with higher priority
        for swarm in swarms:
            for track_id in swarm:
                if track_id in self.tracks and self.tracks[track_id]['active'] and self.can_engage(track_id):
                    detection = self.tracks[track_id]['last_detection']
                    swarm_targets.append({
                        'track_id': track_id,
                        'priority': float('inf'),  # Assign highest priority to swarm members
                        'detection': detection,
                        'track': self.tracks[track_id]
                    })

        # Add non-swarm targets to engageable list
        for track_id, track in self.tracks.items():
            if not track['active']:
                continue
            if not self.can_engage(track_id):
                continue
            if any(track_id in swarm for swarm in swarms):
                continue # Skip tracks that are part of a swarm, already added with high priority

            # Get latest values
            detection = track['last_detection']

            # Calculate priority score for non-swarm targets
            if ENGAGEMENT_PRIORITY == 'distance':
                priority = -detection['distance']  # Closer is higher priority
            elif ENGAGEMENT_PRIORITY == 'confidence':
                priority = detection['confidence']
            elif ENGAGEMENT_PRIORITY == 'velocity':
                priority = track['velocity']
            else:
                priority = -detection['distance']

            engageable.append({
                'track_id': track_id,
                'priority': priority,
                'detection': detection,
                'track': track
            })

        # Combine swarm and non-swarm targets and sort by priority
        all_targets = swarm_targets + engageable
        all_targets.sort(key=lambda x: x['priority'], reverse=True)

        return all_targets[:max_targets]


    def find_swarms(self, tracks):
        """Identify potential swarms based on proximity and minimum members."""
        active_track_ids = [tid for tid, track in tracks.items() if track['active']]
        visited = set()
        swarms = []

        for track_id in active_track_ids:
            if track_id in visited:
                continue

            potential_swarm = {track_id}
            visited.add(track_id)
            queue = deque([track_id])

            while queue:
                current_track_id = queue.popleft()
                current_pos = np.array(tracks[current_track_id]['positions'][-1])

                for other_track_id in active_track_ids:
                    if other_track_id in visited:
                        continue

                    other_pos = np.array(tracks[other_track_id]['positions'][-1])
                    distance = np.linalg.norm(current_pos - other_pos)

                    if distance <= SWARM_MAX_DISTANCE:
                        potential_swarm.add(other_track_id)
                        visited.add(other_track_id)
                        queue.append(other_track_id)

            if len(potential_swarm) >= SWARM_MIN_MEMBERS:
                swarms.append(list(potential_swarm))

        return swarms

class YOLODroneDetector:
    def __init__(self, model_path='yolov8n.pt', camera_index=0):
        """Initialize YOLOv8 drone detector with tracking."""
        print("Loading YOLOv8 model...")
        self.model = YOLO(model_path)
        self.cap = cv2.VideoCapture(camera_index)
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_center_x = self.frame_width // 2
        self.tracker = DroneTracker()
        self.swarms = [] # Added to store identified swarms
        print(f"Model loaded. Camera: {self.frame_width}x{self.frame_height}")

    def estimate_distance(self, bbox_width):
        """Estimate distance using similar triangles."""
        if bbox_width == 0:
            return MAX_RANGE
        distance = (KNOWN_DRONE_WIDTH * FOCAL_LENGTH) / bbox_width
        return min(distance, MAX_RANGE)

    def pixel_to_angle(self, pixel_x):
        """Convert pixel x-coordinate to angle."""
        offset = pixel_x - self.frame_center_x
        angle = (offset / self.frame_center_x) * (FOV_HORIZONTAL / 2)
        return angle

    def detect_and_track(self):
        """Detect and track drones."""
        ret, frame = self.cap.read()
        if not ret:
            return [], None # Return None for display_frame if frame read fails

        # Run YOLOv8 inference
        results = self.model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        display_frame = frame.copy()

        detections = []

        # Process YOLO detections
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls[0])
                class_name = self.model.names[cls_id]
                confidence = float(box.conf[0])

                if class_name in TARGET_CLASSES:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    center_x = int((x1 + x2) / 2)
                    center_y = int((y1 + y2) / 2)
                    bbox_width = x2 - x1

                    angle = self.pixel_to_angle(center_x)
                    distance = self.estimate_distance(bbox_width)

                    if MIN_ANGLE <= angle <= MAX_ANGLE:
                        detections.append({
                            'distance': distance,
                            'angle': angle,
                            'confidence': confidence,
                            'class': class_name,
                            'bbox': (int(x1), int(y1), int(x2), int(y2)),
                            'center': (center_x, center_y)
                        })

        # Update tracker
        tracks = self.tracker.update(detections)

        # Find swarms
        self.swarms = self.tracker.find_swarms(tracks) # Store swarms

        # Draw tracks
        for track_id, track in tracks.items():
            if not track['active']:
                continue

            # Draw track history
            points = list(track['positions'])
            for i in range(1, len(points)):
                thickness = int(np.sqrt(TRACK_HISTORY_LENGTH / float(i + 1)) * 2)
                cv2.line(display_frame, points[i-1], points[i], (0, 255, 255), thickness)

            # Draw current bounding box
            x1, y1, x2, y2 = track['bbox']
            color = (0, 255, 0) if self.tracker.can_engage(track_id) else (0, 165, 255) # Green for engageable, Orange otherwise
            # Check if track belongs to a swarm and change color
            if any(track_id in swarm for swarm in self.swarms):
                 color = (255, 0, 255) # Magenta for swarm members

            cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)


            # Draw track info
            center = track['positions'][-1]
            detection = track['last_detection']

            info_text = f"ID:{track_id} | {detection['distance']:.1f}m | {detection['angle']:.1f}°" # Corrected variable name
            cv2.putText(display_frame, info_text, (x1, y1 - 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            status_text = f"Frames:{track['frames_tracked']} | V:{track['velocity']:.1f}"
            cv2.putText(display_frame, status_text, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

            # Draw crosshair on target
            cv2.drawMarker(display_frame, center, color, cv2.MARKER_CROSS, 20, 2)

        # Draw swarm bounding boxes (optional, for visualization)
        for swarm in self.swarms:
             if len(swarm) > 0:
                 min_x, min_y, max_x, max_y = float('inf'), float('inf'), float('-inf'), float('-inf')
                 for track_id in swarm:
                     if track_id in tracks and tracks[track_id]['active']:
                         x1, y1, x2, y2 = tracks[track_id]['bbox']
                         min_x = min(min_x, x1)
                         min_y = min(min_y, y1)
                         max_x = max(max_x, x2)
                         max_y = max(max_y, y2)
                 if min_x != float('inf'):
                    cv2.rectangle(display_frame, (int(min_x), int(int(min_y))), (int(max_x), int(max_y)), (255, 255, 0), 3) # Yellow for swarm bounding box
                    cv2.putText(display_frame, "SWARM", (int(min_x), int(min_y) - 40),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)


        return tracks, display_frame


    def release(self):
        """Release resources."""
        self.cap.release()
        cv2.destroyAllWindows()

def aim_and_track(target_info):
    """Aim at moving target with predictive tracking."""
    detection = target_info['detection']
    track = target_info['track']

    # Predict future position based on velocity
    if len(track['positions']) >= 2:
        positions = list(track['positions'])
        recent_velocity = np.array(positions[-1]) - np.array(positions[-2])
        # Predict 5 frames ahead
        predicted_pos = np.array(positions[-1]) + (recent_velocity * 5)

        print(f"  → Tracking ID:{target_info['track_id']} with predictive aim")
        print(f"    Position: {detection['distance']:.2f}m, {detection['angle']:.2f}°")
        print(f"    Velocity: {track['velocity']:.1f} px/frame")

    time.sleep(0.1)
    return True

def fire_bazooka(track_id):
    """Fire at tracked target."""
    print(f"  🚀 ENGAGING TARGET ID:{track_id}!")
    time.sleep(0.2)
    return True

def engage_swarm(swarm_tracks, detector_instance):
    """Engage a swarm by targeting the swarm center."""
    print(f"  🚨 ENGAGING SWARM with {len(swarm_tracks)} members!")

    if not swarm_tracks:
        print("  ⚠ Cannot engage empty swarm.")
        return

    # Calculate the centroid of the swarm members' positions
    swarm_center_x = int(np.mean([track['positions'][-1][0] for track in swarm_tracks]))
    swarm_center_y = int(np.mean([track['positions'][-1][1] for track in swarm_tracks]))

    # For simplicity, we'll use the average distance and angle of the swarm members
    # A more sophisticated approach would involve geometric calculations based on the centroid
    swarm_distance = np.mean([track['distances'][-1] for track in swarm_tracks])
    swarm_angle = detector_instance.pixel_to_angle(swarm_center_x)

    print(f"  → Aiming at swarm center: {swarm_distance:.2f}m, {swarm_angle:.2f}°")

    # In a real system, send commands to servo controller to aim at swarm_angle
    # Example using serial communication:
    try:
        servo = serial.Serial('/dev/ttyUSB0', 9600)
        servo.write(f"ANGLE:{swarm_angle}\n".encode())
        servo.close()
    except serial.SerialException as e:
        print(f"  ⚠ Serial communication error: {e}")

    time.sleep(0.5) # Simulate aiming time

    # Fire at the swarm center
    fire_bazooka("SWARM") # Use a special ID for swarm engagement


def main():
    print("=" * 70)
    print("YOLOv8 MULTI-TARGET TRACKING & ENGAGEMENT SYSTEM")
    print("=" * 70)
    print(f"Tracking: {TRACK_HISTORY_LENGTH} frames | Min track: {MIN_TRACK_FRAMES}")
    print(f"Max targets: {MAX_SIMULTANEOUS_TARGETS} | Priority: {ENGAGEMENT_PRIORITY}")
    print(f"Cooldown: {COOLDOWN_TIME}s | Angle tolerance: {ANGLE_TOLERANCE}°")
    print(f"Swarm min members: {SWARM_MIN_MEMBERS} | Swarm max distance: {SWARM_MAX_DISTANCE}") # Added swarm params
    print("\nPress 'q' to quit")
    print("=" * 70)

    detector = YOLODroneDetector(model_path='yolov8n.pt', camera_index=0)

    frame_count = 0
    fps_start = time.time()
    fps = 0

    try:
        while True:
            # Detect and track
            tracks, display_frame = detector.detect_and_track()

            # Calculate FPS
            frame_count += 1
            if frame_count % 30 == 0:
                fps = 30 / (time.time() - fps_start)
                fps_start = time.time()

            # Draw UI
            if display_frame is not None: # Only draw if frame is valid
              cv2.line(display_frame, (detector.frame_center_x, 0),
                      (detector.frame_center_x, detector.frame_height), (255, 0, 0), 2)

              # Check if tracks is a dictionary before accessing values()
              if isinstance(tracks, dict):
                  active_tracks = sum(1 for t in tracks.values() if t['active'])
              else:
                  active_tracks = 0 # Or handle appropriately if tracks is a list

              # Status overlay
              cv2.rectangle(display_frame, (0, 0), (500, 110), (0, 0, 0), -1) # Increased size for swarm info
              cv2.putText(display_frame, f"Active Tracks: {active_tracks} | FPS: {fps:.1f}",
                         (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
              cv2.putText(display_frame, f"Priority: {ENGAGEMENT_PRIORITY.upper()}",
                         (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
              cv2.putText(display_frame, f"Swarms: {len(detector.swarms)}", # Added swarm count
                         (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
              cv2.putText(display_frame, "YOLOv8 Multi-Target System",
                         (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)


              cv2_imshow(display_frame)

            # Get priority targets
            if isinstance(tracks, dict):
                priority_targets = detector.tracker.get_priority_targets(detector.swarms) # Pass swarms to prioritization
            else:
                priority_targets = []

            # Engage targets
            if priority_targets:
                print(f"\n[ENGAGEMENT QUEUE] {len(priority_targets)} target(s):")
                for i, target in enumerate(priority_targets, 1):
                    track_id = target['track_id']
                    det = target['detection']
                    is_swarm_member = any(track_id in swarm for swarm in detector.swarms)

                    print(f"  {i}. ID:{track_id} - {det['class']} | "
                          f"{det['distance']:.2f}m, {det['angle']:.2f}° | "
                          f"Priority: {target['priority']:.2f} | {'SWARM MEMBER' if is_swarm_member else 'SINGLE TARGET'}")

                    if is_swarm_member:
                        # Engage the entire swarm (currently placeholder)
                        # In a real system, you'd target the center or a specific member based on strategy
                        swarm_to_engage = next(swarm for swarm in detector.swarms if track_id in swarm)
                        if all(detector.tracker.can_engage(tid) for tid in swarm_to_engage): # Only engage if all members are engageable
                             # Pass the detector instance to engage_swarm
                             engage_swarm([detector.tracker.tracks[tid] for tid in swarm_to_engage], detector)
                             for tid in swarm_to_engage:
                                 detector.tracker.mark_engaged(tid) # Mark all swarm members as engaged
                    else:
                         # Engage single target
                        if aim_and_track(target):
                            fire_bazooka(track_id)
                            detector.tracker.mark_engaged(track_id)

            else:
                if frame_count % 30 == 0:
                    print("Scanning... No engageable targets")


            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except KeyboardInterrupt:
        print("\n\nShutting down...")
    finally:
        detector.release()
        print("System offline.")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targets
Scanning... No engageable targe

### Develop Swarm Engagement Strategy

Now we need to decide how the system should engage a detected swarm. Here are some potential strategies:

*   **Target the swarm center**: Calculate the centroid of the bounding boxes of all swarm members and aim for that point.
*   **Target the closest member**: Prioritize and engage the individual drone within the swarm that is closest to the system.
*   **Target the densest area**: Identify the area within the swarm with the highest concentration of drones and aim there.
*   **Sequential targeting**: Engage swarm members one by one based on a defined order (e.g., closest first, highest confidence first).
*   **Area suppression**: If the system has a weapon capable of affecting an area (e.g., a net gun or a directed energy weapon), aim for the center or densest area to disable multiple drones simultaneously.

Which strategy would you like to implement first, or do you have another strategy in mind?

### Develop Swarm Engagement Strategy

Now we need to decide how the system should engage a detected swarm. Here are some potential strategies:

*   **Target the swarm center**: Calculate the centroid of the bounding boxes of all swarm members and aim for that point.
*   **Target the closest member**: Prioritize and engage the individual drone within the swarm that is closest to the system.
*   **Target the densest area**: Identify the area within the swarm with the highest concentration of drones and aim there.
*   **Sequential targeting**: Engage swarm members one by one based on a defined order (e.g., closest first, highest confidence first).
*   **Area suppression**: If the system has a weapon capable of affecting an area (e.g., a net gun or a directed energy weapon), aim for the center or densest area to disable multiple drones simultaneously.

Which strategy would you like to implement first, or do you have another strategy in mind?